# Welcome to Colab!

Step 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier

Step 2: load data


In [ ]:
train = pd.read_csv('traindata.csv')
test = pd.read_csv('testdata.csv')

print("Train Shape:", train.shape)
print("Test Shape:", test.shape)

print(train.head())

Train Shape: (614, 13)
Test Shape: (367, 12)
    Loan_ID Gender Married Dependents     Education Self_Employed  \
0  LP001002   Male      No          0      Graduate            No   
1  LP001003   Male     Yes          1      Graduate            No   
2  LP001005   Male     Yes          0      Graduate           Yes   
3  LP001006   Male     Yes          0  Not Graduate            No   
4  LP001008   Male      No          0      Graduate            No   

   ApplicantIncome  CoapplicantIncome  LoanAmount  Loan_Amount_Term  \
0             5849                0.0         NaN             360.0   
1             4583             1508.0       128.0             360.0   
2             3000                0.0        66.0             360.0   
3             2583             2358.0       120.0             360.0   
4             6000                0.0       141.0             360.0   

   Credit_History Property_Area Loan_Status  
0             1.0         Urban           Y  
1             1.0    

Step 3: Check Missing Values

In [ ]:
print(train.isnull().sum())
print(test.isnull().sum())

Loan_ID               0
Gender               13
Married               3
Dependents           15
Education             0
Self_Employed        32
ApplicantIncome       0
CoapplicantIncome     0
LoanAmount           22
Loan_Amount_Term     14
Credit_History       50
Property_Area         0
Loan_Status           0
dtype: int64
Loan_ID               0
Gender               11
Married               0
Dependents           10
Education             0
Self_Employed        23
ApplicantIncome       0
CoapplicantIncome     0
LoanAmount            5
Loan_Amount_Term      6
Credit_History       29
Property_Area         0
dtype: int64


Step 4: Fill Missing Values

In [ ]:
for col in train.columns:
    if train[col].dtype == 'object':
        train[col] = train[col].fillna(train[col].mode()[0])
    else:
        train[col] = train[col].fillna(train[col].median())

for col in test.columns:
    if test[col].dtype == 'object':
        test[col] = test[col].fillna(test[col].mode()[0])
    else:
        test[col] = test[col].fillna(test[col].median())

Step 5: Encode Categorical Columns

In [ ]:
combined = pd.concat([train.drop('Loan_Status', axis=1), test])

categorical_cols = combined.select_dtypes(include='object').columns

for col in categorical_cols:
    le = LabelEncoder()
    combined[col] = le.fit_transform(combined[col].astype(str))

train_encoded = combined.iloc[:len(train), :]
test_encoded = combined.iloc[len(train):, :]

train['Loan_ID'] = train_encoded['Loan_ID']
train['Gender'] = train_encoded['Gender']
train['Married'] = train_encoded['Married']
train['Dependents'] = train_encoded['Dependents']
train['Education'] = train_encoded['Education']
train['Self_Employed'] = train_encoded['Self_Employed']
train['Property_Area'] = train_encoded['Property_Area']

test = test_encoded.copy()

Step 6: Encode Target Column

In [ ]:
target_encoder = LabelEncoder()

train['Loan_Status'] = target_encoder.fit_transform(train['Loan_Status'])

print(train['Loan_Status'].unique())

[1 0]


Step 7: Create Features and Target

In [ ]:
X_train = train.drop(['Loan_ID','Loan_Status'], axis=1)

y_train = train['Loan_Status']

X_test = test.drop('Loan_ID', axis=1)

Step 8: Train Model

In [ ]:
model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

print("Model Trained Successfully")

Model Trained Successfully


Step 9: Predict Loan Status

In [ ]:
predictions = model.predict(X_test)

print(predictions[:10])

[1 1 1 1 0 1 1 0 1 1]


Step 10: Convert Predictions to Approved / Not Approved

In [ ]:
predictions_text = target_encoder.inverse_transform(predictions)

for i in range(10):
    print(predictions_text[i])

Y
Y
Y
Y
N
Y
Y
N
Y
Y


Step 11: Create Final Output File

In [ ]:
submission = pd.DataFrame({
    'Loan_ID': test['Loan_ID'],
    'Loan_Status': predictions_text
})

submission.to_csv('loan_predictions.csv', index=False)

print("Prediction file created successfully!")

Prediction file created successfully!


Step 12: Download Result File

In [ ]:
from google.colab import files

files.download('loan_predictions.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>